# Skip-Connection CAE Training

Trains a Skip-Connection Convolutional Autoencoder (U-Net style) for MVTec AD anomaly detection.

**Key Feature:** Encoder-to-decoder skip connections preserve spatial detail,
producing sharper reconstructions and more precise anomaly localization than vanilla CAE.

**Architecture:**
- Encoder: Input → Conv blocks → (skip connections stored) → Bottleneck
- Decoder: Bottleneck → (concat skip connections) → TransConv blocks → Output

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
sys.path.insert(0, 'F:/Thesis')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving
import numpy as np
import pandas as pd
import time
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix

from src.config import DEVICE, MODELS_DIR, FIGURES_DIR, OUTPUTS_DIR, MVTEC_CATEGORIES, ensure_dirs
from src.data import create_mvtec_dataloaders
from src.data.transforms import denormalize
from src.models.skip_cae import create_skip_cae
from src.training import get_optimizer, get_scheduler, EarlyStopping

ensure_dirs()
print(f"Device: {DEVICE}")

Device: cpu


## Configuration

In [2]:
CONFIG = {
    'batch_size': 16,
    'num_epochs': 50,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'patience': 5,
    'save_every': 10,
}

# Train only bottle category (change as needed)
CATEGORIES_TO_TRAIN = ['bottle']
# CATEGORIES_TO_TRAIN = MVTEC_CATEGORIES  # Uncomment to train all categories

print(f"Categories to train: {CATEGORIES_TO_TRAIN}")
print(f"Epochs: {CONFIG['num_epochs']}")
print(f"Learning rate: {CONFIG['learning_rate']}")

Categories to train: ['bottle']
Epochs: 50
Learning rate: 0.001


## Model Architecture Overview

In [3]:
model_preview = create_skip_cae()
total_params = sum(p.numel() for p in model_preview.parameters())
trainable_params = sum(p.numel() for p in model_preview.parameters() if p.requires_grad)
print(f"Skip-CAE Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Verify input/output shape
test_input = torch.randn(1, 3, 256, 256)
test_output = model_preview(test_input)
print(f"  Input shape:  {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
del model_preview, test_input, test_output

Skip-CAE Architecture:
  Total parameters: 4,304,471
  Trainable parameters: 4,304,471
  Input shape:  torch.Size([1, 3, 256, 256])
  Output shape: torch.Size([1, 3, 256, 256])


## Visualization Helpers

In [4]:
def save_and_show(fig, path):
    """Save figure to disk and display it."""
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"  Saved: {path}")


def plot_loss_curve(history, category, save_path):
    """Plot and save training loss curve."""
    fig, ax = plt.subplots(figsize=(8, 5))
    epochs = range(1, len(history['train_loss']) + 1)
    ax.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Training Loss')
    if 'lr' in history:
        ax2 = ax.twinx()
        ax2.plot(epochs, history['lr'], 'r--', linewidth=1, alpha=0.5, label='Learning Rate')
        ax2.set_ylabel('Learning Rate', color='red')
        ax2.legend(loc='upper right')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.set_title(f'Skip-CAE Training Loss — {category}', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    save_and_show(fig, save_path)


def plot_reconstructions(model, test_loader, category, device, save_path, n_samples=6):
    """Plot original vs reconstruction vs error map for test samples."""
    model.eval()
    images, masks, labels = next(iter(test_loader))
    images = images.to(device)

    with torch.no_grad():
        recons = model(images)
        error_maps = model.get_anomaly_map(images)
        scores = model.get_anomaly_score(images)

    n = min(n_samples, len(images))
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        orig = denormalize(images[i].cpu()).permute(1, 2, 0).numpy().clip(0, 1)
        rec = recons[i].cpu().permute(1, 2, 0).numpy().clip(0, 1)
        err = error_maps[i, 0].cpu().numpy()
        err_norm = (err - err.min()) / (err.max() - err.min() + 1e-8)
        lbl = 'Anomaly' if labels[i].item() == 1 else 'Normal'
        sc = scores[i].item()

        axes[i, 0].imshow(orig)
        axes[i, 0].set_title(f'Original [{lbl}]', fontsize=10)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(rec)
        axes[i, 1].set_title('Reconstruction', fontsize=10)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(err_norm, cmap='hot')
        axes[i, 2].set_title(f'Error Map (score={sc:.4f})', fontsize=10)
        axes[i, 2].axis('off')

        axes[i, 3].imshow(orig)
        axes[i, 3].imshow(err_norm, cmap='jet', alpha=0.5)
        axes[i, 3].set_title('Heatmap Overlay', fontsize=10)
        axes[i, 3].axis('off')

    fig.suptitle(f'Skip-CAE Reconstructions — {category}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    save_and_show(fig, save_path)


def plot_score_distribution(all_scores, all_labels, category, save_path):
    """Plot anomaly score distribution for normal vs anomalous samples."""
    scores_arr = np.array(all_scores)
    labels_arr = np.array(all_labels)
    normal_scores = scores_arr[labels_arr == 0]
    anomaly_scores = scores_arr[labels_arr == 1]

    fig, ax = plt.subplots(figsize=(8, 5))
    if len(normal_scores) > 0:
        ax.hist(normal_scores, bins=30, alpha=0.6, label=f'Normal (n={len(normal_scores)})', color='green')
    if len(anomaly_scores) > 0:
        ax.hist(anomaly_scores, bins=30, alpha=0.6, label=f'Anomaly (n={len(anomaly_scores)})', color='red')
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Count')
    ax.set_title(f'Skip-CAE Score Distribution — {category}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_and_show(fig, save_path)


def plot_roc_curve(all_scores, all_labels, auc_val, category, save_path):
    """Plot ROC curve."""
    fpr, tpr, _ = roc_curve(all_labels, all_scores)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'Skip-CAE (AUC = {auc_val:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {category}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    save_and_show(fig, save_path)

## Training Function

In [5]:
def train_skip_cae_category(category):
    print(f"\n{'='*60}")
    print(f"Training Skip-CAE: {category.upper()}")
    print(f"{'='*60}")

    # 1. Load Data
    try:
        train_loader, test_loader = create_mvtec_dataloaders(
            category, batch_size=CONFIG['batch_size'], return_mask=True
        )
    except Exception as e:
        print(f"Skipping {category}: {e}")
        return None

    print(f"  Train batches: {len(train_loader)}")
    print(f"  Test batches:  {len(test_loader)}")

    # 2. Create Model
    model = create_skip_cae().to(DEVICE)
    optimizer = get_optimizer(model, lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    scheduler = get_scheduler(optimizer, patience=2, factor=0.5)
    early_stopping = EarlyStopping(patience=CONFIG['patience'], mode='min')
    criterion = nn.MSELoss()

    # 3. Training Loop
    history = {'train_loss': [], 'lr': []}
    start_time = time.time()

    for epoch in tqdm(range(1, CONFIG['num_epochs'] + 1), desc=f'{category}'):
        model.train()
        epoch_loss = 0.0

        for batch in train_loader:
            images = batch[0].to(DEVICE)
            optimizer.zero_grad()
            reconstruction = model(images)
            loss = criterion(reconstruction, images)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        scheduler.step(avg_loss)

        if early_stopping(avg_loss):
            print(f"Early stopping at epoch {epoch}")
            break

    train_time = time.time() - start_time
    print(f"  Training completed in {train_time:.1f}s ({train_time/60:.1f}min)")

    # 4. Evaluate
    model.eval()
    all_scores, all_labels = [], []
    with torch.no_grad():
        for img, mask, label in test_loader:
            img = img.to(DEVICE)
            scores = model.get_anomaly_score(img)
            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(label.numpy())

    try:
        auc = roc_auc_score(all_labels, all_scores)
        print(f"  {category.upper()} ROC-AUC: {auc:.4f}")
    except:
        auc = 0.0

    # 5. Save model
    save_path = MODELS_DIR / f'skip_cae_{category}_final.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': CONFIG,
        'history': history,
        'auc': auc,
    }, save_path)
    print(f"  Model saved: {save_path}")

    # 6. Generate all visualizations
    print("  Generating visualizations...")

    # 6a. Training loss curve
    plot_loss_curve(history, category, FIGURES_DIR / f'skip_cae_{category}_loss.png')

    # 6b. Reconstructions
    plot_reconstructions(model, test_loader, category, DEVICE,
                         FIGURES_DIR / f'skip_cae_{category}_reconstructions.png')

    # 6c. Score distribution
    plot_score_distribution(all_scores, all_labels, category,
                            FIGURES_DIR / f'skip_cae_{category}_scores.png')

    # 6d. ROC curve
    if auc > 0:
        plot_roc_curve(all_scores, all_labels, auc, category,
                       FIGURES_DIR / f'skip_cae_{category}_roc.png')

    return {
        'category': category,
        'auc': auc,
        'final_loss': history['train_loss'][-1],
        'epochs_trained': len(history['train_loss']),
        'train_time_s': round(train_time, 1),
    }

## Run Training

In [6]:
results = []

for category in CATEGORIES_TO_TRAIN:
    result = train_skip_cae_category(category)
    if result is not None:
        results.append(result)

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")


Training Skip-CAE: BOTTLE
  Train batches: 14
  Test batches:  6


bottle:  74%|███████▍  | 37/50 [48:28<17:01, 78.60s/it] 

Early stopping at epoch 38
  Training completed in 2908.1s (48.5min)


  BOTTLE ROC-AUC: 0.4095
  Model saved: F:\Thesis\outputs\models\skip_cae_bottle_final.pth
  Generating visualizations...


C:\Users\hamim\AppData\Local\Temp\ipykernel_14172\308446269.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved: F:\Thesis\outputs\figures\skip_cae_bottle_loss.png
  Saved: F:\Thesis\outputs\figures\skip_cae_bottle_reconstructions.png
  Saved: F:\Thesis\outputs\figures\skip_cae_bottle_scores.png
  Saved: F:\Thesis\outputs\figures\skip_cae_bottle_roc.png

TRAINING COMPLETE


C:\Users\hamim\AppData\Local\Temp\ipykernel_14172\308446269.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results Summary

In [7]:
if results:
    df = pd.DataFrame(results)
    df = df.sort_values('auc', ascending=False)
    print("\nSkip-CAE Results:")
    print(df.to_string(index=False))
    print(f"\nMean AUC: {df['auc'].mean():.4f}")

    # Save results table as CSV
    csv_path = OUTPUTS_DIR / 'skip_cae_results.csv'
    df.to_csv(csv_path, index=False)
    print(f"Results saved: {csv_path}")

    # Bar chart of AUC per category
    if len(df) > 1:
        fig, ax = plt.subplots(figsize=(10, 5))
        colors = ['green' if a >= 0.7 else 'orange' if a >= 0.5 else 'red' for a in df['auc']]
        ax.barh(df['category'], df['auc'], color=colors)
        ax.set_xlabel('ROC-AUC')
        ax.set_title('Skip-CAE Performance by Category', fontsize=14, fontweight='bold')
        ax.set_xlim([0, 1])
        ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
        for i, v in enumerate(df['auc']):
            ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='x')
        save_and_show(fig, FIGURES_DIR / 'skip_cae_auc_summary.png')
else:
    print("No results to display.")


Skip-CAE Results:
category      auc  final_loss  epochs_trained  train_time_s
  bottle 0.409524    0.004437              38        2908.1

Mean AUC: 0.4095
Results saved: F:\Thesis\outputs\skip_cae_results.csv
